In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
X = pd.read_csv("../data/X_clean.csv")
Y = pd.read_csv("../data/Y_clean.csv").values.ravel()

### Machine Learning Implementation

In [3]:
from sklearn.model_selection import train_test_split

In [4]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.20,random_state=42)

In [5]:
X_train.shape

(5634, 30)

In [6]:
X_test.shape

(1409, 30)

In [7]:
Y_train.shape

(5634,)

In [8]:
Y_test.shape

(1409,)

In [9]:
from sklearn.ensemble import RandomForestClassifier

In [10]:
rf_model=RandomForestClassifier(n_estimators=100,random_state=42)

In [11]:
rf_model.fit(X_train,Y_train)

RandomForestClassifier(random_state=42)

In [12]:
y_pred=rf_model.predict(X_test)
y_pred

array([0, 0, 0, ..., 1, 0, 1], dtype=int64)

In [13]:
Y_test

array([1, 0, 0, ..., 0, 0, 1], dtype=int64)

In [14]:
from sklearn.metrics import accuracy_score

In [15]:
accuracy=accuracy_score(Y_test,y_pred)
print(accuracy)

0.7856635911994322


In [16]:
from sklearn.metrics import confusion_matrix

In [17]:
cm=confusion_matrix(Y_test,y_pred)
print(cm)

[[902 107]
 [195 205]]


In [18]:
from sklearn.metrics import classification_report
print(classification_report(Y_test,y_pred))

              precision    recall  f1-score   support

           0       0.82      0.89      0.86      1009
           1       0.66      0.51      0.58       400

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.72      1409
weighted avg       0.78      0.79      0.78      1409



**Approach 1- Handle Class Imbalance**

In [19]:
rf_balanced=RandomForestClassifier(n_estimators=100,random_state=42,class_weight='balanced')
rf_balanced.fit(X_train,Y_train)
y_pred_balanced=rf_balanced.predict(X_test)
accuracy_balanced=accuracy_score(Y_test,y_pred_balanced)
cm_balanced=confusion_matrix(Y_test,y_pred_balanced)
print(accuracy_balanced,cm_balanced)
print(classification_report(Y_test,y_pred_balanced))

0.7920511000709723 [[907 102]
 [191 209]]
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1009
           1       0.67      0.52      0.59       400

    accuracy                           0.79      1409
   macro avg       0.75      0.71      0.72      1409
weighted avg       0.78      0.79      0.78      1409



**Approach 2- Hyperparameter Tuning**

In [20]:
rf_tuned=RandomForestClassifier(n_estimators=300,max_depth=10,random_state=42,class_weight='balanced')
rf_tuned.fit(X_train,Y_train)
y_pred_tuned=rf_tuned.predict(X_test)
accuracy_tuned=accuracy_score(Y_test,y_pred_tuned)
cm_tuned=confusion_matrix(Y_test,y_pred_tuned)
print(accuracy_tuned,cm_tuned)
print(classification_report(Y_test,y_pred_tuned))

0.7828246983676366 [[804 205]
 [101 299]]
              precision    recall  f1-score   support

           0       0.89      0.80      0.84      1009
           1       0.59      0.75      0.66       400

    accuracy                           0.78      1409
   macro avg       0.74      0.77      0.75      1409
weighted avg       0.80      0.78      0.79      1409



**Approach 3- Feature Importance Analysis**

In [21]:
import pandas as pd
feature_importance =pd.DataFrame({
    'Features':X.columns,
    'Importance':rf_tuned.feature_importances_
})
feature_importance=feature_importance.sort_values(by='Importance',ascending=False)
print(feature_importance)

                                  Features  Importance
0                            Tenure Months    0.179069
2                            Total Charges    0.136460
25                       Contract_Two year    0.102861
1                          Monthly Charges    0.093681
6                           Dependents_Yes    0.068517
10            Internet Service_Fiber optic    0.060228
28         Payment Method_Electronic check    0.046732
24                       Contract_One year    0.036682
13                     Online Security_Yes    0.029356
19                        Tech Support_Yes    0.021106
26                   Paperless Billing_Yes    0.018064
5                              Partner_Yes    0.015846
12     Online Security_No internet service    0.014973
18        Tech Support_No internet service    0.014799
22    Streaming Movies_No internet service    0.014360
16   Device Protection_No internet service    0.012985
11                     Internet Service_No    0.012673
3         

In [22]:
print(feature_importance.tail(15))

                                  Features  Importance
16   Device Protection_No internet service    0.012985
11                     Internet Service_No    0.012673
3                              Gender_Male    0.012586
15                       Online Backup_Yes    0.012213
9                       Multiple Lines_Yes    0.011690
14       Online Backup_No internet service    0.010298
4                       Senior Citizen_Yes    0.010257
20        Streaming TV_No internet service    0.010186
23                    Streaming Movies_Yes    0.010161
27  Payment Method_Credit card (automatic)    0.009430
21                        Streaming TV_Yes    0.009289
17                   Device Protection_Yes    0.009101
29             Payment Method_Mailed check    0.008800
7                        Phone Service_Yes    0.003872
8          Multiple Lines_No phone service    0.003725


In [23]:
X_selected=X.drop(['Phone Service_Yes','Multiple Lines_No phone service'],axis=1)

In [24]:
X_train_sel,X_test_sel,Y_train_sel,Y_test_sel=train_test_split(X_selected,Y,test_size=0.20,random_state=42)

In [25]:
rf_selected=RandomForestClassifier(n_estimators=300,max_depth=10,random_state=42,class_weight='balanced')
rf_selected.fit(X_train_sel,Y_train_sel)
y_pred_selected=rf_selected.predict(X_test_sel)

print(classification_report(Y_test_sel,y_pred_selected))

              precision    recall  f1-score   support

           0       0.89      0.80      0.84      1009
           1       0.60      0.74      0.66       400

    accuracy                           0.78      1409
   macro avg       0.74      0.77      0.75      1409
weighted avg       0.80      0.78      0.79      1409



**Approach 2 Again-Combination of trees and depth**

In [26]:
from sklearn.metrics import accuracy_score,recall_score,precision_score,f1_score

In [27]:
n_estimators_list=[100,200,300,400,500]
max_depth_list=[5,10,15,20]
results=[]
for n_trees in n_estimators_list:
    for depth in max_depth_list:
        rf=RandomForestClassifier(n_estimators=n_trees,max_depth=depth,random_state=42,class_weight='balanced')
        rf.fit(X_train,Y_train)
        y_pred=rf.predict(X_test)
        accuracy=accuracy_score(Y_test,y_pred)
        recall=recall_score(Y_test,y_pred)
        precision=precision_score(Y_test,y_pred)
        f1=f1_score(Y_test,y_pred)
        results.append({'Trees': n_trees, 'Depth':depth, 'Accuracy':accuracy, 'Recall':recall, 'precision':precision,'F1 Scrore':f1})
result_df=pd.DataFrame(results)
result_df=result_df.sort_values(by=['Recall','Accuracy'],ascending=False)
print(result_df.head(20))

    Trees  Depth  Accuracy  Recall  precision  F1 Scrore
0     100      5  0.745919  0.8200   0.534202   0.646943
8     300      5  0.743790  0.8200   0.531605   0.645034
12    400      5  0.743790  0.8125   0.531915   0.642928
4     200      5  0.742370  0.8100   0.530278   0.640950
16    500      5  0.740951  0.8050   0.528736   0.638256
1     100     10  0.775727  0.7525   0.581081   0.655773
9     300     10  0.782825  0.7475   0.593254   0.661504
13    400     10  0.782825  0.7475   0.593254   0.661504
17    500     10  0.781405  0.7475   0.590909   0.660044
5     200     10  0.782115  0.7450   0.592445   0.660022
10    300     15  0.806246  0.6400   0.664935   0.652229
14    400     15  0.803407  0.6375   0.658915   0.648030
18    500     15  0.801987  0.6375   0.655527   0.646388
6     200     15  0.804826  0.6350   0.663185   0.648787
2     100     15  0.802697  0.6300   0.659686   0.644501
3     100     20  0.787793  0.5325   0.655385   0.587586
7     200     20  0.793471  0.5

In [28]:
from sklearn.model_selection import cross_val_score
final_rf=RandomForestClassifier(n_estimators=300,max_depth=10,random_state=42,class_weight='balanced')

In [29]:
cv_accuracy=cross_val_score(final_rf,X,Y,cv=5,scoring='accuracy')
cv_accuracy

array([0.76721079, 0.79772889, 0.76224273, 0.78551136, 0.78409091])

In [30]:
cv_accuracy.mean()

0.7793569343183432

In [31]:
cv_recall=cross_val_score(final_rf,X,Y,cv=5,scoring='recall')
cv_recall

array([0.70855615, 0.76470588, 0.74064171, 0.74331551, 0.71045576])

In [32]:
cv_recall.mean()

0.7335350030823931

**OPTIONAL(ROC-AUC)**

In [33]:
from sklearn.metrics import roc_auc_score,roc_curve

In [34]:
import matplotlib.pyplot as plt

In [35]:
y_prob=rf_tuned.predict_proba(X_test)

In [36]:
churn_prob=y_prob[:,1]
fpr,tpr,threshold=roc_curve(Y_test,churn_prob)
auc_score=roc_auc_score(Y_test,churn_prob)
print(auc_score)

0.857104806739346
